In [1]:
# =========================
# Install Required Library
# =========================

!pip install gradio -q

In [2]:
# =========================
# Import Libraries
# =========================

import gradio as gr
import cv2
import numpy as np
from tensorflow.keras.models import load_model
from google.colab import files

In [3]:
# =========================
# Upload and Load Trained Model
# =========================

uploaded = files.upload()

model = load_model("microstructure_unet_model.h5", compile=False)

print("Model loaded successfully.")

Saving microstructure_unet_model.h5 to microstructure_unet_model.h5
Model loaded successfully.


In [4]:
# =========================
# Define Image Size, Phase Colors and Class Names
# =========================

IMG_SIZE = 256

colors = {
    0: [0, 0, 0],          # Background
    1: [250, 0, 0],        # Ferrite
    2: [215, 227, 121],    # Martensite
    3: [61, 245, 61]       # Pearlite
}

class_names = {
    0: "Background",
    1: "Ferrite",
    2: "Martensite",
    3: "Pearlite"
}

In [5]:
# =========================
# Define Image Analysis Function
# =========================

def analyze_microstructure(image):
    # Resize image to match model input size
    image_resized = cv2.resize(image, (IMG_SIZE, IMG_SIZE))

    # Normalize pixel values from 0-255 to 0-1
    image_normalized = image_resized / 255.0

    # Add batch dimension and predict segmentation mask
    prediction = model.predict(np.expand_dims(image_normalized, axis=0))

    # Convert prediction probabilities into class IDs
    predicted_mask = np.argmax(prediction[0], axis=-1)

    # Create empty RGB mask
    colored_mask = np.zeros((IMG_SIZE, IMG_SIZE, 3), dtype=np.uint8)

    # Assign color to each predicted class
    for class_id, color in colors.items():
        colored_mask[predicted_mask == class_id] = color

    # Blend original image with predicted mask
    overlay = (0.6 * image_resized + 0.4 * colored_mask).astype(np.uint8)

    # Calculate predicted phase percentages
    percentages = ""

    for class_id in np.unique(predicted_mask):
        percent = np.sum(predicted_mask == class_id) / predicted_mask.size * 100
        percentages += f"{class_names[class_id]}: {percent:.2f}%\n"

    return colored_mask, overlay, percentages

In [6]:
# =========================
# Build and Launch GUI Interface
# =========================

demo = gr.Interface(
    fn=analyze_microstructure,
    inputs=gr.Image(type="numpy", label="Upload Microstructure Image"),
    outputs=[
        gr.Image(type="numpy", label="Predicted Phase Mask"),
        gr.Image(type="numpy", label="Overlay Image"),
        gr.Textbox(label="Detected Phase Percentages")
    ],
    title="Microstructure Analysis Tool",
    description=(
        "Upload a microstructure image to generate a predicted phase mask, "
        "overlay image, and estimated phase percentages."
    )
)

demo.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://24405ae50cc887f499.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [7]:
#Install
#↓
#Import
#↓
#Load trained model
#↓
#Define colors/classes
#↓
#Analyze uploaded image
#↓
#Launch GUI